In [1]:
"""
Ingests resolved tickets from data/tickets.db into the 'tickets' Chroma collection.
Run once (or after adding new tickets): python ingest_tickets.py
"""
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import sqlite3
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
CHROMA_DIR = "chroma_store"
COLLECTION  = "tickets"
DB_PATH     = os.path.join("data", "tickets.db")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [3]:
def load_ticket_documents(db_path: str) -> list[Document]:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT * FROM tickets WHERE status = 'resolved'"
    ).fetchall()
    conn.close()

    docs = []
    for row in rows:
        # Combine issue description + resolution into a single searchable text block
        content = (
            f"Issue: {row['issue_type']}\n"
            f"Description: {row['description']}\n"
            f"Resolution: {row['resolution']}"
        )
        docs.append(Document(
            page_content=content,
            metadata={
                "source":    "ticket",
                "ticket_id": row["ticket_id"],
                "category":  row["category"],
                "status":    row["status"],
            },
        ))
    return docs

In [4]:
print("Loading ticket documents from SQLite...")
docs = load_ticket_documents(DB_PATH)
print(f"  {len(docs)} resolved tickets loaded.")

print("Initialising embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

print(f"Embedding and storing in Chroma collection '{COLLECTION}'...")
vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=CHROMA_DIR,
    )
print(f"  Done. {vectorstore._collection.count()} vectors stored.")

Loading ticket documents from SQLite...
  19 resolved tickets loaded.
Initialising embedding model...


d:\GitClones_new\AgenticAI_Study\codeBasics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2832.84it/s]


Embedding and storing in Chroma collection 'tickets'...
  Done. 19 vectors stored.
